# Lesson 5.2: Sentiment Analysis with Hugging Face (RoBERTa)

## Overview

In Lesson 5.1 you used **VADER**, a fast rule-based tool, to score sentiment. In this lesson you will use **RoBERTa**, a transformer-based language model fine-tuned on Twitter data, and compare its results to VADER.

By the end of this lesson you will have:
- Loaded and run the `cardiffnlp/twitter-roberta-base-sentiment` model from Hugging Face
- Applied it to the same geoparsed Reddit sentences from Lesson 5.1
- Compared the two methods side-by-side and mapped emotional sentiment to geographic locations

RoBERTa (Robustly Optimized BERT Pretraining Approach) is available on [Hugging Face](https://huggingface.co/). Check out other sentiment models [here](https://huggingface.co/models?sort=trending&search=sentiment).

---

---

## 1 Running RoBERTa

While running Vader was not particurally complicated RoBERTa requires a bit more finesse. Because this requires knowing concepts outside the scope of this course and some helper functions, all the more complicated logic has been tucked into `sentiment_utils.py`. In the code below, we are first going to recreate our Vader Sentiment scores from the previous lesson and then we are going to create a sample column of RoBERTa scores. Because RoBERTA takes a long time to run, you want to avoid running it constantly. Indeed, for the final project you should only run it if you find that some of the locations are off and need to be fixed

In [1]:
import pandas as pd
from sentiment_utils import add_sentiment_to_column
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

# Once your Google Sheet is published as CSV, replace the URL below.
# In Google Sheets: File > Share > Publish to web > select CSV format
SHEET_URL = "https://docs.google.com/spreadsheets/d/YOUR_SHEET_ID/export?format=csv"

# Fallback paths (tried in order if the Google Sheet is unavailable)
BACKUP_CSV    = "../data/JMU/JMU_geoparsed_long_backup.csv"
BACKUP_PICKLE = "../data/JMU/JMU_geoparsed_long_backup_sentiment.pickle"

try:
    df_reddit_geoparsed_long = pd.read_csv(SHEET_URL)
    print("✅ Loaded from Google Sheet")
except Exception:
    print("⚠️  Google Sheet not available — trying backup CSV")
    try:
        df_reddit_geoparsed_long = pd.read_csv(BACKUP_CSV, dtype=str).fillna('')
        print(f"✅ Loaded from backup CSV ({len(df_reddit_geoparsed_long):,} rows)")
    except FileNotFoundError:
        print("⚠️  Backup CSV not found — loading from local pickle")
        df_reddit_geoparsed_long = pd.read_pickle(BACKUP_PICKLE)
        print(f"✅ Loaded from pickle ({len(df_reddit_geoparsed_long):,} rows)")

df_reddit_geoparsed_long['vader_sentiment'] = df_reddit_geoparsed_long['sentences'].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)
print(f"✅ Sentiment scored — {len(df_reddit_geoparsed_long):,} rows total")

df_reddit_sentiment_sample = add_sentiment_to_column(
    df_reddit_geoparsed_long, "sentences", num_rows=100
)

print("✅ Processing complete")


⚠️  Google Sheet not available — trying backup CSV
✅ Loaded from backup CSV (884 rows)
✅ Sentiment scored — 884 rows total


Processing Sentiment Analysis: 100%|██████████| 100/100 [00:05<00:00, 18.77it/s]

✅ Processing complete


### Evaluate the Sample

While the analysis only tagged the emotions for 100 sentences this should be enough to see the differences between the two taggers. The output below looks at where the two taggers disagree and pulls a random sample.


In [2]:
RANDOM_STATE = 42  # ← change this to see different contradictions

# Filter to rows where VADER and RoBERTa disagree significantly, then sample
df_reddit_sentiment_sample['disagreement'] = (
    df_reddit_sentiment_sample['vader_sentiment'] - df_reddit_sentiment_sample['roberta_compound']
).abs()

contradictions = df_reddit_sentiment_sample[df_reddit_sentiment_sample['disagreement'] > 0.3]

(
    contradictions[['sentences', 'vader_sentiment', 'roberta_compound']]
    .sample(4, random_state=RANDOM_STATE)
    .style
    .set_properties(subset=['sentences'], **{'white-space': 'normal', 'width': '500px'})
    .format({'vader_sentiment': '{:.3f}', 'roberta_compound': '{:.3f}'})
)

,sentences,vader_sentiment,roberta_compound
85,It seems like standard procedures were ignored or they are lacking in Harrisonburg.,0.052,-0.814
56,Be smart and stay away from Harrisonburg in the fall.,0.402,-0.055
58,"I mean to be fair, be super careful about any solicitation in Harrisonburg.",0.778,-0.060
91,"Compared to other public universities in Virginia, JMU has stagnated a bit.",0.000,-0.301


> 💡 **Critical Reflection:**
> 1. How did the models do? 
> 2. Which appears to be more accurate? Why do you think that is?
> 3. Are there sentences where you do not think either emotional evaluation is accurate?
> Cycle through the samples by changing `random_state=` to a different integer and identify a sentence where the language model does particularly well or poorly.

---

## 2 Creating the Full Dataset

Running RoBERTa on all sentences takes a long time. The line below is intentionally **commented out**. When you are ready, one group member should remove the `#`, run this cell, then complete Section 7 to save and share the result.


In [ ]:
# df_reddit_sentiment_full = add_sentiment_to_column(df_reddit_geoparsed_long, 'sentences')

### Restore Backup

If you skipped the full run (or restarted the kernel), load the version your teammate already committed to the repository.


In [3]:
try:
    df_reddit_sentiment_full
    print("✅ df_reddit_sentiment_full is already loaded in memory")
except NameError:
    print("📥 Loading from shared data folder...")
    try:
        df_reddit_sentiment_full = pd.read_pickle('../data/jmu_reddit_sentiment_full.pickle')
        print(f"✅ Loaded {len(df_reddit_sentiment_full):,} rows")
    except FileNotFoundError:
        try:
            df_reddit_sentiment_full = pd.read_pickle(
                '../data/JMU/JMU_geoparsed_long_backup_sentiment.pickle'
            )
            print(f"⚠️  Teammate file not found — loaded instructor backup ({len(df_reddit_sentiment_full):,} rows)")
            print("   This is pre-scored data from the instructor's backup CSV.")
            print("   Your group still needs to run and commit the full dataset (Sections 6–7) to replace this.")
        except FileNotFoundError:
            print("❌ Neither the team file nor the instructor backup was found.")
            print("   Ask the teammate who ran the full analysis to complete Section 7 first.")


📥 Loading from shared data folder...
⚠️  Teammate file not found — loaded instructor backup (884 rows)
   This is pre-scored data from the instructor's backup CSV.
   Your group still needs to run and commit the full dataset (Sections 6–7) to replace this.


In [4]:
df_reddit_sentiment_full.sample(n=10, random_state=42)

,type,date,score,sentences,year_month,place,latitude,longitude,feature_type,admin1_name,...,corrected_name,corrected_place_type,reviewer,place_count,corrected_latlon,vader_sentiment,roberta_neg,roberta_neu,roberta_pos,roberta_compound
44,comment,13-03-20,87,"But in all seriousness, no party is worth enda...",2020-03,City of Harrisonburg,38.44957,-78.86892,second-order administrative division,Virginia,...,Harrisonburg,City,Joost,87,"38.44957, -78.86892",0.7227,0.822920,0.167243,0.009837,-0.677100
388,post,27-05-22,68,"Photos of Mrs. Green's, a lunch operation that...",2022-05,Chandler Hall,38.30263,-77.4761,building(s),Virginia,...,Chandler Hall,Building,Joost,6,"38.43317533, -78.87362232",0.3182,0.013969,0.950783,0.035248,0.001047
638,comment,22-06-21,13,My suggestion is to wander your way up south m...,2021-06,Pheasant Run,40.31011,-75.106,,Pennsylvania,...,,,,2,,0.0000,0.018194,0.857697,0.124109,0.015072
739,comment,31-10-23,9,The top floors of EnGeo and King hall are alwa...,2023-10,Engeo,53.47235,9.13243,populated place,Lower Saxony,...,,,,1,,0.5719,0.002915,0.081956,0.915129,0.837452
879,comment,12-02-20,4,So they make in a single semester about the sa...,2020-02,Algiers,36.73225,3.08746,capital of a political entity,Algiers,...,,,,1,,0.0000,0.195804,0.760814,0.043382,-0.036457
839,post,26-11-23,110,THE JMU DUKES ARE GOING BOWLING After the loss...,2023-11,South Carolina,34.00043,-81.00009,first-order administrative division,South Carolina,...,,,,1,,-0.6239,0.005500,0.277135,0.717365,0.514582
333,comment,07-08-20,1,You wanted crap in your room on the 7th floor ...,2020-08,Basalt,39.36887,-107.03282,populated place,Colorado,...,Eagle Hall (JMU),Building,Joost,8,"38.43399862, -78.87297345",-0.7351,0.961603,0.035843,0.002554,-0.924674
300,comment,24-01-24,2,You mentioned the Southview fire - I'm both su...,2024-01,Southview,51.03333,-113.98333,section of populated place,Alberta,...,The Hills Southview Apartments,Building,Joost,13,"38.41687425, -78.87158834",-0.2715,0.052805,0.229192,0.718003,0.512740
120,comment,17-10-20,6,Just saw on the thread in Virginia subreddit t...,2020-10,Virginia,37.54812,-77.44675,first-order administrative division,Virginia,...,,,,77,,-0.5423,0.327668,0.655811,0.016521,-0.107094
859,comment,11-03-20,3,"What make this even worse is if we define ""bul...",2020-03,University Park,27.39044,-82.46883,populated place,Florida,...,,,,1,,0.0258,0.827149,0.161155,0.011696,-0.684038


---

## 3 Create and Share the Dataset

**One group member only** should complete this section.

You have run RoBERTa over the full dataset. Before moving to the next lesson, save that result as a **pickle** file and commit it to the shared repository. This way the rest of your team can load it without re-running the model (which takes a long time).

> ⚠️ Only one person should commit this file. If two people commit different versions of the pickle at the same time you will get a merge conflict on a binary file, which cannot be resolved automatically.


In [ ]:
df_reddit_sentiment_full = add_sentiment_to_column(df_reddit_geoparsed_long, 'sentences')
df_reddit_sentiment_full.to_pickle('../data/jmu_reddit_sentiment_full.pickle')
print(f"✅ Saved {len(df_reddit_sentiment_full):,} rows to data/jmu_reddit_sentiment_full.pickle")

### Commit and Push

Now commit this file so your teammates can access it. Follow the same **branch → commit → pull request** workflow from [Lesson 1.1](../lesson_1_the_team/lesson_1_1_git_and_pull_requests.ipynb).

1. **Create a new branch** — e.g. `firstname-sentiment-data`
2. Open **Source Control** (<img src="../lesson_assets/images/vscode/source-control.svg" alt="Source Control icon" width="14">) and confirm only `data/jmu_reddit_sentiment_full.pickle` appears under **Changes**
   - If any other file appears, click the **Discard Changes** icon next to it
3. Stage the file (`+`), then use the ✨ sparkle button to auto-generate a commit message
4. Click the dropdown next to **Commit** and select **Commit & Sync**
5. Click **Publish Branch**, then open a Pull Request with base `main`
6. Merge the PR and delete the branch
7. **Everyone else:** switch back to `main` and click **Sync Changes** to download the file

✅ The pickle file is now in the root `data/` folder — the shared handoff folder that Lesson 6 reads from.


---

## Lesson Summary

Here is what you covered in this lesson:

### Part 1: RoBERTa — Transformer-Based Sentiment
- **RoBERTa** — a transformer model fine-tuned on Twitter data; understands context across the whole sentence rather than word by word
- **`AutoTokenizer` / `AutoModelForSequenceClassification`** — Hugging Face classes for loading pre-trained models
- **`softmax()`** — converts raw model scores into probabilities that sum to 1
- **`roberta_compound`** — a derived score: `(pos - neg) × (1 - neu)`, ranging from −1 to +1

### Part 2: Comparing Methods
- **Contradictions** — sentences where VADER and RoBERTa disagree most reveal each model's blind spots
- **Edge cases** — comparing outputs at the extremes exposes the assumptions built into each model

### Part 3: Sharing Your Work
- **Pickle** — a binary format that preserves the full DataFrame so teammates can load it without re-running the model
- The **branch → PR workflow** from Lesson 1.1 ensures only one clean version of the data lands in `main`

---

➡️ **Next:** [Lesson 6 — Mapping Fundamentals](../lesson_6_mapping_fundamentals/lesson_6_mapping_fundamentals.ipynb)
